In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

Scrape data about [UGA Football Record](https://www.sicemdawgs.com/uga-football-records/vs-all-opponents/) from `https://www.sicemdawgs.com/uga-football-records/vs-all-opponents/`

- What is the first team that UGA played against
- What are the most challenging opponents?

In [2]:
response = requests.get("https://www.sicemdawgs.com/uga-football-records/vs-all-opponents/")

In [3]:
response.status_code

200

In [4]:
soup = BeautifulSoup(response.text, 'html')

In [5]:
rows = soup.find_all('tr')

In [6]:
def process_row(row):
    try:
        values = row.find_all('td')
        opponent = values[0].text
        firstgame = values[1].text
        lastgame = values[2].text
        games = values[3].text
        win = values[4].text
        lost = values[5].text
        tie = values[6].text
        return {"Opponent":opponent, "First Game":firstgame, "Last Game":lastgame,
                "G": games, "W":win, "L": lost, "T":tie}
    except Exception as error:
        print("Cannot process row " + str(row))
        return None
    

In [7]:
data = [process_row(row) for row in rows]

Cannot process row <tr>
<th class="schgry1">Opponent</th>
<th class="schgry2">First Game</th>
<th class="schgry2">Last Game</th>
<th class="schgry2">G</th>
<th class="schgry2">W</th>
<th class="schgry2">L</th>
<th class="schgry2">T</th>
</tr>
Cannot process row <tr>
<td class="ros2" colspan="7">**Georgia discredits losses to Georgia Tech in 1943 (48-0) and 1944 (44-0) due to WWII.</td>
</tr>
Cannot process row <tr>
<td class="ros2" colspan="7">**Formerly known as Oklahoma A&amp;M.</td>
</tr>


In [8]:
data = [d for d in data if d is not None]

In [9]:
df = pd.DataFrame(data)

In [10]:
df.dtypes

Opponent      object
First Game    object
Last Game     object
G             object
W             object
L             object
T             object
dtype: object

In [11]:
df=df[["Opponent", "First Game", "Last Game", "G", "W", "L", "T"]].set_index("Opponent")

In [12]:
df.dtypes

First Game    object
Last Game     object
G             object
W             object
L             object
T             object
dtype: object

In [13]:
df = df.map(pd.to_numeric) #convert all values to numbers

In [14]:
df.head()

,First Game,Last Game,G,W,L,T
Opponent,,,,,,
Alabama,1895,2024,74,26,44,4
Alabama Presbyterian,1911,1913,2,2,0,0
Appalachian State,2013,2017,2,2,0,0
Arizona,1985,1985,1,0,0,1
Arizona State,2008,2009,2,2,0,0


In [15]:
#What is the first team that UGA played against
first_year = df.sort_values("First Game").iloc[0, 0]
df[df['First Game']==first_year]

,First Game,Last Game,G,W,L,T
Opponent,,,,,,
Auburn,1892,2024,129,65,56,8
Mercer,1892,1941,22,22,0,0


In [16]:
list(df[df['First Game']==first_year].index)

['Auburn', 'Mercer']

In [17]:
# What are the most challenging opponents?
# define challenging as hard to win against
df['difficulty'] = df['L'] / (df['W'] + df['L'])
df.sort_values('difficulty', ascending=False)

,First Game,Last Game,G,W,L,T,difficulty
Opponent,,,,,,,
Cumberland,1905,1905,1,0,1,0,1.0
Stanford,1978,1978,1,0,1,0,1.0
"Miami, Ohio",1974,1974,1,0,1,0,1.0
Navy,1916,1957,2,0,2,0,1.0
Holy Cross,1937,1939,3,0,3,0,1.0
...,...,...,...,...,...,...,...
Idaho State,2010,2010,1,1,0,0,0.0
Duke,1986,1986,1,1,0,0,0.0
Arizona,1985,1985,1,0,0,1,NaN


In [18]:
df.sort_values(['G', 'difficulty'], ascending=False)

,First Game,Last Game,G,W,L,T,difficulty
Opponent,,,,,,,
Auburn,1892,2024,129,65,56,8,0.462810
Georgia Tech,1893,2024,116,72,39,5,0.351351
Florida,1904,2024,103,57,44,2,0.435644
Vanderbilt,1893,2023,83,61,20,2,0.246914
Kentucky,1939,2024,78,64,12,2,0.157895
...,...,...,...,...,...,...,...
William & Mary,1988,1988,1,1,0,0,0.000000
Wyoming,1998,1998,1,1,0,0,0.000000
Arizona,1985,1985,1,0,0,1,NaN


In [19]:
# or weigh difficulty by the number of games
df['challenging'] = df['G'] * df['difficulty']
df.sort_values('challenging', ascending=False)

,First Game,Last Game,G,W,L,T,difficulty,challenging
Opponent,,,,,,,,
Auburn,1892,2024,129,65,56,8,0.462810,59.702479
Alabama,1895,2024,74,26,44,4,0.628571,46.514286
Florida,1904,2024,103,57,44,2,0.435644,44.871287
Georgia Tech,1893,2024,116,72,39,5,0.351351,40.756757
Tennessee,1899,2024,54,29,23,2,0.442308,23.884615
...,...,...,...,...,...,...,...,...
Louisiana Lafayette,2010,2016,2,2,0,0,0.000000,0.000000
Newberry,1915,1922,2,2,0,0,0.000000,0.000000
Arizona,1985,1985,1,0,0,1,NaN,NaN
